# 01_build_model_base_data

This notebook isolates the **data extraction + label creation + baseline feature joining steps** for the prediction task.

**Inputs**
*   Raw JSON / Delta tables for course, user, interaction, and problem

**Outputs**
*   `MODEL_BASE_TABLE` containing:
    *   Primary keys: `user_id`, `course_id`
    *   Labels: `completion_rate`, `is_completed`
    *   All raw features prior to the data leakage filtering step

> This design aligns with your current flow: creating `model_df` by joining `base_enroll_df` with `user_features_df`, `course_features_df`, `user_course_features_df`, and `label_df`.

In [2]:
import requests

StatementMeta(, d4e431da-af47-4bfb-9dd0-76ce781dc4eb, 3, Finished, Available, Finished, False)

In [3]:
#Danh sách URL của các tệp dữ liệu cần thiết (chỉ các bảng liên quan đến features trong sheet)
file_urls = {
    "course.json":            "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/course.json",
    "video.json":             "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/video.json",
    "user.json":              "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/user.json",
    "user-video.json":        "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/user-video.json",
    "user-problem.json":      "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/user-problem.json",
    "problem.json":           "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/problem.json",
    "comment.json":           "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/comment.json",
    "reply.json":             "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/reply.json",
    "concept-video.txt":      "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/concept-video.txt",
    "concept-course.txt":     "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/concept-course.txt",
    "exercise-problem.txt":   "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/exercise-problem.txt",
    "concept-problem.txt":    "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/concept-problem.txt",
    "concept-comment.txt":    "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/concept-comment.txt",
    "concept-other.txt":      "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/relations/concept-other.txt",
    "concept.json":           "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/concept.json",
}

# Tải từng tệp về thư mục "Files" trên Lakehouse
for filename, url in file_urls.items():
    print(f"Downloading {filename} ...")
    resp = requests.get(url)
    resp.raise_for_status()
    with open(f"/lakehouse/default/Files/temp/{filename}", "wb") as f:
        f.write(resp.content)
print("Tải dữ liệu hoàn tất.")

StatementMeta(, d4e431da-af47-4bfb-9dd0-76ce781dc4eb, 4, Finished, Available, Finished, False)

Tải dữ liệu hoàn tất.


In [ ]:
# =========================
# 0) CONFIG
# =========================
RAW_ROOT = "Files/temp"   # đổi lại theo workspace của bạn
MODEL_BASE_TABLE = "dbo.tbl_model_base_course_completion"

COMPLETION_THRESHOLD = 0.60
SCORE_PCT_THRESHOLD = 0.01
SESSION_GAP_MINUTES = 30

StatementMeta(, 90b29f07-066b-4c95-b3ce-d7d73aa93572, -1, Cancelled, , Cancelled, True)

In [8]:
# =========================
# 1) IMPORTS
# =========================
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 10, Finished, Available, Finished, False)

In [9]:
# =========================
# 2) HELPERS
# =========================
def normalize_colnames(df, default_prefix="No_name"):
    seen = set()
    new_names = []
    for i, c in enumerate(df.columns):
        name = "" if c is None else str(c).strip()
        if name == "":
            name = f"{default_prefix}_{i}"
        name = name.replace("`", "")
        name = re.sub(r"[^0-9a-zA-Z_]", "_", name)
        name = re.sub(r"_+", "_", name).strip("_")
        if name == "":
            name = f"{default_prefix}_{i}"
        base = name
        k = 1
        while name in seen:
            name = f"{base}_{k}"
            k += 1
        seen.add(name)
        new_names.append(name)
    return df.toDF(*new_names)


def ensure_prefixed(df, col_name, prefix):
    return df.withColumn(
        col_name,
        F.when(
            F.col(col_name).cast("string").startswith(prefix),
            F.col(col_name).cast("string")
        ).otherwise(F.concat(F.lit(prefix), F.col(col_name).cast("string")))
    )


def safe_divide(numerator_col, denominator_col):
    return F.when(denominator_col.isNull() | (denominator_col == 0), F.lit(0.0)) \
            .otherwise(numerator_col / denominator_col)

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 11, Finished, Available, Finished, False)

In [10]:
# =========================
# 3) LOAD DATA
# =========================
# ---- Option A: đọc từ raw JSON ----
course_raw = spark.read.json(f"{RAW_ROOT}/course.json")
user_raw = spark.read.json(f"{RAW_ROOT}/user.json")
comment_raw = spark.read.json(f"{RAW_ROOT}/comment.json")
reply_raw = spark.read.json(f"{RAW_ROOT}/reply.json")
user_video_raw = spark.read.json(f"{RAW_ROOT}/user-video.json")
user_problem_raw = spark.read.json(f"{RAW_ROOT}/user-problem.json")
problem_raw = spark.read.json(f"{RAW_ROOT}/problem.json")
exercise_problem_raw = spark.read.csv(
    f"{RAW_ROOT}/exercise-problem.txt",
    sep="\t",
    schema="exercise_id STRING, problem_id STRING"
)

display(course_raw.limit(3))
display(user_raw.limit(3))

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5a2312bd-578c-4424-a49a-a979d463b686)

SynapseWidget(Synapse.DataFrame, ac943c85-14cf-4372-8a96-549847229a0d)

In [16]:
# =========================
# 4) PREPARE ENTITY TABLES
# =========================
course_df = (
    course_raw
    .select(
        F.col("id").alias("course_id"),
        F.col("name").alias("course_name"),
        F.col("about"),
        F.col("prerequisites")
    )
    .dropDuplicates(["course_id"])
)

course_field_df = course_raw.select(
    F.col("id").alias("course_id"),
    F.explode(F.col("field")).alias("field_id")
).dropDuplicates(["course_id", "field_id"])

course_resource_df = course_raw.select(
    F.col("id").alias("course_id"),
    F.explode(F.col("resource")).alias("resource")
).select(
    F.col("course_id"),
    F.col("resource.resource_id"),
    F.col("resource.chapter"),
    F.explode(F.col("resource.titles")).alias("title")
).dropDuplicates(["course_id", "resource_id", "title"])

user_df = (
    user_raw
    .select(
        F.col("id").alias("user_id"),
        F.col("gender"),
        F.col("year_of_birth"),
        F.col("school").alias("school_id")
    )
    .dropDuplicates(["user_id"])
)

# enroll_time[i] tương ứng course_order[i]
user_course_enroll_df = user_raw.select(
    F.col("id").alias("user_id"),
    F.posexplode(F.col("course_order")).alias("idx", "course_index")
).join(
    user_raw.select(
        F.col("id").alias("uid"),
        F.col("enroll_time")
    ),
    F.col("user_id") == F.col("uid"),
    "left"
).select(
    F.col("user_id"),
    F.col("course_index"),
    F.col("enroll_time")[F.col("idx")].alias("enroll_time")
).dropDuplicates(["user_id", "course_index"])

user_course_enroll_df = ensure_prefixed(user_course_enroll_df, "course_index", "C_")


comment_df = spark.read.json("Files/temp/comment.json").select(
    F.col("id").alias("comment_id"),
    F.col("user_id").cast("string"),
    F.col("resource_id"),
    F.col("text").alias("comment_text"),
    F.col("create_time")
).dropna(subset=["comment_id"]).dropDuplicates(["comment_id"])

reply_df = reply_raw.select("user_id").dropDuplicates()

user_video_df = user_video_raw.select(
    F.col("user_id"),
    F.explode(F.col("seq")).alias("watch")
).select(
    F.col("user_id"),
    F.col("watch.video_id")
).dropna(subset=["user_id", "video_id"]).dropDuplicates()

user_problem_df = (
    user_problem_raw
    .select(
        "user_id",
        "problem_id",
        "is_correct",
        "score"
    )
    .dropDuplicates(["user_id", "problem_id"])
)
user_problem_df = ensure_prefixed(user_problem_df, "problem_id", "Pm_")

df_problem = problem_raw
df_problem = ensure_prefixed(df_problem, "problem_id", "Pm_")

exercise_problem_df = (
    exercise_problem_raw
    .select("exercise_id", "problem_id")
    .dropDuplicates(["exercise_id", "problem_id"])
)

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 18, Finished, Available, Finished, False)

In [19]:
# =========================
# 5) FEATURE BUILDERS
# =========================
def build_user_core_features(
    user_df,
    user_video_df,
    user_problem_df,
    user_course_enroll_df,
    comment_df,
    reply_df,
    course_field_df
):
    videos_per_user = (
        user_video_df
        .groupBy("user_id")
        .agg(F.countDistinct("video_id").alias("num_videos_watched"))
    )

    problem_stats = (
        user_problem_df
        .groupBy("user_id")
        .agg(
            F.countDistinct("problem_id").alias("num_problems_attempted"),
            F.avg(F.col("is_correct").cast("int")).alias("accuracy")
        )
    )

    courses_enrolled = (
        user_course_enroll_df
        .groupBy("user_id")
        .agg(F.countDistinct("course_index").alias("num_courses_enrolled"))
    )

    user_comment_count = (
        comment_df.groupBy("user_id").agg(F.count("*").alias("num_comments"))
    )

    user_reply_count = (
        reply_df.groupBy("user_id").agg(F.count("*").alias("num_replies"))
    )

    user_fields_df = (
        user_course_enroll_df
        .join(course_field_df, user_course_enroll_df.course_index == course_field_df.course_id, "left")
        .groupBy("user_id")
        .agg(F.countDistinct("field_id").alias("num_fields_enrolled"))
    )

    score_per_user = (
        user_problem_df
        .groupBy("user_id")
        .agg(
            F.avg(F.col("score").cast("double")).alias("avg_score"),
            F.sum(F.col("score").cast("double")).alias("total_score")
        )
    )

    correct_count_df = (
        user_problem_df
        .filter(F.col("is_correct") == True)
        .groupBy("user_id")
        .agg(F.count("*").alias("num_correct"))
    )

    return (
        user_df
        .join(videos_per_user, on="user_id", how="left")
        .join(problem_stats, on="user_id", how="left")
        .join(courses_enrolled, on="user_id", how="left")
        .join(user_comment_count, on="user_id", how="left")
        .join(user_reply_count, on="user_id", how="left")
        .join(user_fields_df, on="user_id", how="left")
        .join(score_per_user, on="user_id", how="left")
        .join(correct_count_df, on="user_id", how="left")
        .fillna({
            "num_videos_watched": 0,
            "num_problems_attempted": 0,
            "accuracy": 0.0,
            "num_courses_enrolled": 0,
            "num_comments": 0,
            "num_replies": 0,
            "num_fields_enrolled": 0,
            "avg_score": 0.0,
            "total_score": 0.0,
            "num_correct": 0
        })
    )


def build_user_time_features(user_video_raw, session_gap_minutes: int = 30):
    # Explode segments 1 lần để reuse
    seg_df = (
        user_video_raw
        .select("user_id", F.explode("seq").alias("seq"))
        .select("user_id", F.explode("seq.segment").alias("seg"))
    )

    # Total time spent
    # Lưu ý: cần xác nhận đơn vị của end_point/start_point có đúng là phút hay không
    time_spent_df = (
        seg_df
        .withColumn("duration", F.col("seg.end_point") - F.col("seg.start_point"))
        .groupBy("user_id")
        .agg(F.sum("duration").alias("time_spent"))
    )

    # Activity timestamps
    session_df = (
        seg_df
        .withColumn("ts", (F.col("seg.local_start_time") / 1000).cast("timestamp"))
        .dropna(subset=["ts"])
        .select("user_id", "ts")
    )

    window_ts = Window.partitionBy("user_id").orderBy("ts")

    session_df2 = (
        session_df
        .withColumn("prev_ts", F.lag("ts").over(window_ts))
        .withColumn(
            "diff_min",
            F.round((F.col("ts").cast("long") - F.col("prev_ts").cast("long")) / 60, 1)
        )
        .withColumn(
            "new_session",
            F.when(F.col("prev_ts").isNull(), 1)
             .when(F.col("diff_min") > SESSION_GAP_MINUTES, 1)
             .otherwise(0)
        )
    )

    session_count_df = (
        session_df2
        .groupBy("user_id")
        .agg(F.sum("new_session").alias("session_count"))
    )

    daily_activity_df = (
        session_df2
        .withColumn("day", F.to_date("ts"))
        .groupBy("user_id", "day")
        .agg(F.count("*").alias("num_actions"))
    )

    # Trajectory: The difference in the number of actions between the two most recent active days.
    window_day_desc = Window.partitionBy("user_id").orderBy(F.col("day").desc())

    trajectory_df = (
        daily_activity_df
        .withColumn("rank", F.row_number().over(window_day_desc))
        .filter(F.col("rank") <= 2)
        .groupBy("user_id")
        .pivot("rank")
        .agg(F.first("num_actions"))
        .withColumn(
            "trajectory_diff",
            F.coalesce(F.col("1"), F.lit(0)) - F.coalesce(F.col("2"), F.lit(0))
        )
        .select("user_id", "trajectory_diff")
    )

    # Moving average 2 nearest active date
    ma2_window = Window.partitionBy("user_id").orderBy("day").rowsBetween(-1, 0)

    ma2_df = (
        daily_activity_df
        .withColumn("ma2", F.avg("num_actions").over(ma2_window))
    )

    ma2_final_df = (
        ma2_df
        .withColumn("rank", F.row_number().over(window_day_desc))
        .filter(F.col("rank") == 1)
        .select("user_id", "ma2")
    )

    # Variance / skewness activity
    variance_df = (
        daily_activity_df
        .groupBy("user_id")
        .agg(
            F.variance("num_actions").alias("activity_variance"),
            F.skewness("num_actions").alias("activity_skewness")
        )
    )

    # Days active
    days_active_df = (
        session_df2
        .withColumn("active_day", F.to_date("ts"))
        .groupBy("user_id")
        .agg(F.countDistinct("active_day").alias("days_active"))
    )

    # First / last activity + active span
    activity_span_df = (
        session_df2
        .groupBy("user_id")
        .agg(
            F.min("ts").alias("first_activity_ts"),
            F.max("ts").alias("last_activity_ts")
        )
        .withColumn(
            "active_span_days",
            F.datediff(F.to_date("last_activity_ts"), F.to_date("first_activity_ts")) + F.lit(1)
        )
    )

    # Tổng số actions
    total_actions_df = (
        daily_activity_df
        .groupBy("user_id")
        .agg(F.sum("num_actions").alias("total_actions"))
    )

    # Self-paced frequency features
    frequency_df = (
        days_active_df
        .join(activity_span_df, "user_id", "left")
        .join(total_actions_df, "user_id", "left")
        .withColumn(
            "activity_density",
            F.when(F.col("active_span_days") > 0,
                   F.col("days_active") / F.col("active_span_days"))
             .otherwise(F.lit(0.0))
        )
        .withColumn(
            "avg_actions_per_active_day",
            F.when(F.col("days_active") > 0,
                   F.col("total_actions") / F.col("days_active"))
             .otherwise(F.lit(0.0))
        )
    )

    user_time_features_df = (
        time_spent_df
        .join(session_count_df, "user_id", "left")
        .join(trajectory_df, "user_id", "left")
        .join(variance_df, "user_id", "left")
        .join(ma2_final_df, "user_id", "left")
        .join(frequency_df, "user_id", "left")
        .fillna({
            "time_spent": 0.0,
            "session_count": 0,
            "trajectory_diff": 0,
            "activity_variance": 0.0,
            "activity_skewness": 0.0,
            "ma2": 0.0,
            "days_active": 0,
            "active_span_days": 0,
            "total_actions": 0,
            "activity_density": 0.0,
            "avg_actions_per_active_day": 0.0
        })
    )

    return user_time_features_df


def build_course_features(course_df, course_field_df, course_resource_df, user_course_enroll_df):
    field_count_df = (
        course_field_df.groupBy("course_id").agg(F.countDistinct("field_id").alias("num_fields"))
    )
    resource_count_df = (
        course_resource_df.groupBy("course_id").agg(F.countDistinct("resource_id").alias("num_resources"))
    )
    enroll_count_df = (
        user_course_enroll_df
        .groupBy("course_index")
        .agg(F.countDistinct("user_id").alias("num_enrollments"))
        .withColumnRenamed("course_index", "course_id")
    )
    return (
        course_df
        .join(field_count_df, on="course_id", how="left")
        .join(resource_count_df, on="course_id", how="left")
        .join(enroll_count_df, on="course_id", how="left")
        .fillna({"num_fields": 0, "num_resources": 0, "num_enrollments": 0})
    )


def build_completion_status(
    course_resource_df,
    exercise_problem_df,
    user_problem_df,
    problem_df,
    completion_threshold=0.6,
    pass_mode="score_pct",   # "correct_or_positive_score" | "score_pct"
    score_pct_threshold=0.5,
    enrollment_df=None
):
    if "max_score" in problem_df.columns:
        df_max = problem_df.select("problem_id", F.col("max_score").cast("double").alias("max_score"))
    elif "score" in problem_df.columns:
        df_max = problem_df.select("problem_id", F.col("score").cast("double").alias("max_score"))
    else:
        raise ValueError("problem_df must contain 'max_score' or 'score'.")

    course_exercise_df = (
        course_resource_df
        .filter(F.col("resource_id").startswith("Ex_"))
        .select("course_id", F.col("resource_id").alias("exercise_id"))
        .distinct()
    )

    df_course_problem_master = (
        course_exercise_df
        .join(exercise_problem_df, on="exercise_id", how="inner")
        .select("course_id", "problem_id")
        .dropDuplicates(["course_id", "problem_id"])
    )

    df_course_total_problems = (
        df_course_problem_master
        .groupBy("course_id")
        .agg(F.countDistinct("problem_id").alias("total_problems"))
    )

    up = (
        user_problem_df
        .select("user_id", "problem_id", "is_correct", F.col("score").cast("double").alias("score_d"))
    )

    up2 = (
        up.join(df_max, on="problem_id", how="left")
        .withColumn("normalized_score", safe_divide(F.coalesce(F.col("score_d"), F.lit(0.0)), F.col("max_score")))
        .withColumn(
            "is_passed",
            F.when(
                F.lit(pass_mode) == F.lit("correct_or_positive_score"),
                ((F.col("is_correct") == True) | (F.coalesce(F.col("score_d"), F.lit(0.0)) > 0)).cast("int")
            ).otherwise((F.col("normalized_score") >= F.lit(score_pct_threshold)).cast("int"))
        )
    )

    user_course_problem_df = (
        df_course_problem_master
        .join(up2, on="problem_id", how="left")
        .select("user_id", "course_id", "problem_id", "is_passed")
    )

    passed_df = (
        user_course_problem_df
        .groupBy("user_id", "course_id")
        .agg(F.sum(F.coalesce(F.col("is_passed"), F.lit(0))).alias("problems_passed"))
    )

    if enrollment_df is not None:
        base_df = (
            enrollment_df
            .select("user_id", F.col("course_index").alias("course_id"))
            .dropDuplicates(["user_id", "course_id"])
        )
    else:
        base_df = (
            user_course_problem_df
            .select("user_id", "course_id")
            .dropDuplicates(["user_id", "course_id"])
        )

    df_final_completion_status = (
        base_df
        .join(df_course_total_problems, on="course_id", how="left")
        .join(passed_df, on=["user_id", "course_id"], how="left")
        .fillna({"total_problems": 0, "problems_passed": 0})
        .withColumn(
            "completion_rate",
            safe_divide(F.col("problems_passed").cast("double"), F.col("total_problems").cast("double"))
        )
        .withColumn(
            "is_completed",
            F.when(F.col("completion_rate") >= F.lit(completion_threshold), F.lit(1.0)).otherwise(F.lit(0.0))
        )
    )

    return df_final_completion_status, df_course_problem_master, df_course_total_problems


def build_user_course_features(
    user_course_enroll_df,
    course_resource_df,
    user_video_df,
    user_problem_df,
    df_course_problem_master,
    df_problem
):
    base_enroll_df = (
        user_course_enroll_df
        .select("user_id", F.col("course_index").alias("course_id"), "enroll_time")
        .dropDuplicates(["user_id", "course_id"])
    )

    course_video_map = (
        course_resource_df
        .filter(F.col("resource_id").startswith("V_"))
        .select("course_id", F.col("resource_id").alias("video_id"))
        .distinct()
    )

    uc_video_df = (
        user_video_df
        .join(course_video_map, on="video_id", how="inner")
        .groupBy("user_id", "course_id")
        .agg(F.countDistinct("video_id").alias("course_videos_watched"))
    )

    if "max_score" in df_problem.columns:
        df_max = df_problem.select("problem_id", F.col("max_score").cast("double").alias("max_score"))
    else:
        df_max = df_problem.select("problem_id", F.col("score").cast("double").alias("max_score"))

    up2 = (
        user_problem_df
        .select("user_id", "problem_id", "is_correct", F.col("score").cast("double").alias("score_d"))
        .join(df_max, on="problem_id", how="left")
        .withColumn("normalized_score", safe_divide(F.coalesce(F.col("score_d"), F.lit(0.0)), F.col("max_score")))
        .withColumn(
            "is_passed",
            F.when((F.col("is_correct") == True) | (F.col("normalized_score") > 0), F.lit(1)).otherwise(F.lit(0))
        )
    )

    uc_problem_df = (
        df_course_problem_master
        .join(up2, on="problem_id", how="left")
        .groupBy("user_id", "course_id")
        .agg(
            F.countDistinct("problem_id").alias("course_problems_attempted"),
            F.sum("is_passed").alias("course_problems_passed"),
            F.avg("normalized_score").alias("course_total_raw_score")
        )
    )

    return (
        base_enroll_df
        .join(uc_video_df, on=["user_id", "course_id"], how="left")
        .join(uc_problem_df, on=["user_id", "course_id"], how="left")
        .withColumn(
            "course_pass_rate",
            safe_divide(
                F.coalesce(F.col("course_problems_passed").cast("double"), F.lit(0.0)),
                F.coalesce(F.col("course_problems_attempted").cast("double"), F.lit(0.0))
            )
        )
        .fillna({
            "course_videos_watched": 0,
            "course_problems_attempted": 0,
            "course_problems_passed": 0,
            "course_total_raw_score": 0.0,
            "course_pass_rate": 0.0
        })
    )

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 21, Finished, Available, Finished, False)

In [20]:
# =========================
# 6) BUILD LABEL + FEATURES
# =========================
df_final_completion_status, df_course_problem_master, df_course_total_problems = build_completion_status(
    course_resource_df=course_resource_df,
    exercise_problem_df=exercise_problem_df,
    user_problem_df=user_problem_df,
    problem_df=df_problem,
    completion_threshold=COMPLETION_THRESHOLD,
    pass_mode="score_pct",
    score_pct_threshold=SCORE_PCT_THRESHOLD,
    enrollment_df=user_course_enroll_df
)

label_df = df_final_completion_status.select(
    "user_id", "course_id", "completion_rate", "is_completed"
)

user_core_features_df = build_user_core_features(
    user_df=user_df,
    user_video_df=user_video_df,
    user_problem_df=user_problem_df,
    user_course_enroll_df=user_course_enroll_df,
    comment_df=comment_df,
    reply_df=reply_df,
    course_field_df=course_field_df
)

SESSION_GAP_MINUTES = 30

user_time_features_df = build_user_time_features(
    user_video_raw=user_video_raw,
    session_gap_minutes=SESSION_GAP_MINUTES
)

user_features_df = (
    user_core_features_df
    .join(user_time_features_df, on="user_id", how="left")
    .fillna({
        "time_spent": 0.0,                     # nếu bạn dùng tên mới
        # "time_spent_minutes": 0.0,          # dùng dòng này thay thế nếu vẫn giữ tên cũ
        "session_count": 0,
        "trajectory_diff": 0,
        "activity_variance": 0.0,
        "activity_skewness": 0.0,
        "ma2": 0.0,
        "days_active": 0,
        "active_span_days": 0,
        "total_actions": 0,
        "activity_density": 0.0,
        "avg_actions_per_active_day": 0.0
    })
)

course_features_df = build_course_features(
    course_df=course_df,
    course_field_df=course_field_df,
    course_resource_df=course_resource_df,
    user_course_enroll_df=user_course_enroll_df
)

user_course_features_df = build_user_course_features(
    user_course_enroll_df=user_course_enroll_df,
    course_resource_df=course_resource_df,
    user_video_df=user_video_df,
    user_problem_df=user_problem_df,
    df_course_problem_master=df_course_problem_master,
    df_problem=df_problem
)

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 22, Finished, Available, Finished, False)

In [21]:
# =========================
# 7) BUILD MODEL BASE TABLE
# =========================
base_enroll_df = (
    user_course_enroll_df
    .select("user_id", F.col("course_index").alias("course_id"), "enroll_time")
    .dropDuplicates(["user_id", "course_id"])
)

model_df = (
    base_enroll_df
    .join(user_features_df, on="user_id", how="left")
    .join(course_features_df, on="course_id", how="left")
    .join(user_course_features_df.drop("enroll_time"), on=["user_id", "course_id"], how="left")
    .join(label_df, on=["user_id", "course_id"], how="left")
)

numeric_cols = [
    f.name
    for f in model_df.schema.fields
    if f.dataType.simpleString() in ("int", "bigint", "double", "float", "smallint", "tinyint", "decimal")
]

model_df = (
    model_df
    .fillna(0, subset=numeric_cols)
    .fillna({"is_completed": 0.0, "completion_rate": 0.0})
)

print("Rows in model_df:", model_df.count())
display(model_df.limit(10))

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 23, Finished, Available, Finished, False)

Rows in model_df: 11807090


SynapseWidget(Synapse.DataFrame, 56ee517f-9185-4df1-91c9-3ac3dd49c543)

In [24]:
# =========================
# 8) SAVE TO DB
# =========================
spark.sql(f"DROP TABLE IF EXISTS {MODEL_BASE_TABLE}")

(
    model_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(MODEL_BASE_TABLE)
)

print(f"Saved model base table to: {MODEL_BASE_TABLE}")
spark.table(MODEL_BASE_TABLE).printSchema()
display(spark.table(MODEL_BASE_TABLE).limit(10))

StatementMeta(, 7dac776a-7b43-45b0-92da-57f9b23d79e1, 26, Finished, Available, Finished, False)

Saved model base table to: dbo.tbl_model_base_course_completion
root
 |-- user_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- enroll_time: string (nullable = true)
 |-- gender: long (nullable = true)
 |-- year_of_birth: long (nullable = true)
 |-- school_id: string (nullable = true)
 |-- num_videos_watched: long (nullable = true)
 |-- num_problems_attempted: long (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- num_courses_enrolled: long (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- num_replies: long (nullable = true)
 |-- num_fields_enrolled: long (nullable = true)
 |-- avg_score: double (nullable = true)
 |-- total_score: double (nullable = true)
 |-- num_correct: long (nullable = true)
 |-- time_spent: double (nullable = true)
 |-- session_count: long (nullable = true)
 |-- trajectory_diff: long (nullable = true)
 |-- activity_variance: double (nullable = true)
 |-- activity_skewness: double (nullable = true)
 |-- ma2: dou

SynapseWidget(Synapse.DataFrame, bb5bc7c3-1cab-4f14-a958-5718523d8988)